# Image Generation

AIMU ships a text-to-image surface backed by HuggingFace `diffusers` that runs in parallel to the text-chat surface. The two stay distinct because diffusion is a different modality with a different shape (`generate(prompt) -> Image` vs. `chat(message, ...) -> text + tool calls`).

## Install

```bash
pip install -e '.[hf,google]'
```

This pulls `diffusers`, `accelerate`, `safetensors`, `Pillow`, `torch`, and `transformers`.

## Layers of progressive disclosure

1. **One-shot**: `aimu.generate_image(prompt, model=...)` for quick scripts.
2. **Direct client**: `aimu.image_client(model)` reuses loaded weights across calls.
3. **As an agent tool**: `from aimu.tools import builtin; Agent(client, tools=[builtin.generate_image])` lets a chat LLM call image gen when the user asks for one.
4. **As a skill**: a `SKILL.md` adds prompt-engineering guidance on top of the tool for `SkillAgent` users.
5. **Async**: `await aio.image_client(sync_client).generate(...)` — in-process providers wrap an existing sync client so weights aren't loaded twice.

## 1. One-shot generation

In [ ]:
import aimu

image = aimu.generate_image(
    "a watercolor of a fox in a snowy forest, soft morning light",
    model="hf:runwayml/stable-diffusion-v1-5",
    width=512,
    height=512,
    num_inference_steps=20,
    seed=42,
)
image  # displays inline in Jupyter

### Cloud alternative: Google Nano Banana

`gemini:nano-banana` (alias for `gemini-2.5-flash-image`) is Google's image model accessible via the Gemini API. The same `aimu.image_client()` / `aimu.generate_image()` entry points dispatch to it based on the `gemini:` prefix. Requires `GOOGLE_API_KEY` in the environment (or `.env`).

No weights to load — fast first call, but every call hits the cloud. Aspect ratio is the main knob (`"1:1"`, `"16:9"`, `"3:4"`, …).

In [ ]:
# Requires GOOGLE_API_KEY in env or .env.
nb_image = aimu.generate_image(
    "a watercolor of a fox in a snowy forest, soft morning light",
    model="gemini:nano-banana",
    aspect_ratio="16:9",
)
nb_image

## 2. Direct client (reuse loaded weights)

Constructing a fresh client per call reloads weights, which is slow. For repeated generation, build one client and call `generate()` on it.

In [ ]:
from aimu import HuggingFaceImageModel, image_client

client = image_client(HuggingFaceImageModel.SD_1_5)

for prompt in [
    "a watercolor of a fox",
    "a watercolor of a deer",
    "a watercolor of an owl",
]:
    img = client.generate(prompt, width=512, height=512, num_inference_steps=20)
    display(img)

### Output formats

`format=` selects how the image is returned:

- `"pil"` (default) — `PIL.Image` for direct manipulation / display
- `"path"` — saves PNG under `output/images/{timestamp}-{hash}.png` and returns the path
- `"bytes"` — raw PNG bytes
- `"data_url"` — `data:image/png;base64,...` for embedding in HTML / responses

In [ ]:
path = client.generate("a watercolor of a heron in flight", format="path", width=512, height=512)
print(f"Saved to: {path}")

data_url = client.generate("a small icon of a leaf", format="data_url", width=256, height=256)
print(data_url[:80], "...")

## 3. Image generation as an agent tool

The built-in `generate_image` tool lets any chat LLM produce images when the user asks. The LLM decides *when* to call it; the tool returns a saved file path which becomes part of the conversation history.

The first call loads weights (slow); subsequent calls reuse the singleton.

In [ ]:
import os

# Optional: pick the singleton's default model via env var.
os.environ["AIMU_IMAGE_MODEL"] = "hf:runwayml/stable-diffusion-v1-5"

from aimu import client
from aimu.agents import Agent
from aimu.tools import builtin

text_client = client("anthropic:claude-sonnet-4-6")

agent = Agent(
    text_client,
    system_message=(
        "You can generate images using the generate_image tool. When the user asks for an image, "
        "craft a vivid prompt and call the tool. Tell the user where the image was saved."
    ),
    tools=[builtin.generate_image],
)

response = agent.run("Please make me a watercolor painting of a fox curled up in a snowy forest.")
print(response)

### Per-agent model — `make_image_tool`

If you don't want the singleton (e.g. each agent should use a different diffusion model), use `make_image_tool` to build a tool bound to a specific client.

In [ ]:
from aimu.tools.builtin import make_image_tool

fast_client = image_client(HuggingFaceImageModel.FLUX_SCHNELL)  # 4-step model, very fast
fast_image_tool = make_image_tool(fast_client)

agent = Agent(text_client, tools=[fast_image_tool])
agent.run("Sketch me a cyberpunk fox.")

## 4. As a skill — deeper integration with `SkillAgent`

A skill layers *instructions* on top of the tool. Create `.agents/skills/image-generation/SKILL.md`:

```markdown
---
name: image-generation
description: Generate images using a diffusion model. Use this when the user asks for a picture, illustration, sketch, or photo.
---

## When to use

Activate this skill when the user requests an image, illustration, sketch, photo, drawing, or visual.

## Prompt engineering for diffusion

When calling `generate_image`, write descriptive prompts that include:

1. **Subject** — what is in the image
2. **Style** — watercolor, oil painting, photograph, line drawing, 3D render, anime, etc.
3. **Composition** — close-up, wide shot, low angle, top-down, etc.
4. **Lighting** — soft morning light, golden hour, studio lighting, dramatic shadows
5. **Detail level** — "highly detailed", "minimalist", "painterly"

Combine 2-4 descriptors per prompt. More than that often hurts coherence.

## Tools

- `generate_image(prompt)` — returns the saved file path.
```

Then use `SkillAgent` — it auto-discovers `SKILL.md` files and injects the catalog.

In [ ]:
from aimu.agents import SkillAgent
from aimu.tools import builtin

# SkillAgent will discover the SKILL.md you placed under .agents/skills/.
skill_agent = SkillAgent(text_client, tools=[builtin.generate_image])
skill_agent.run("I want a sketch of a robot fixing a vintage typewriter.")

## 5. Async surface

The async surface mirrors the sync surface one-for-one. Because diffusers loads model weights in-process, the async factory wraps an existing sync client rather than re-loading weights — passing a `HuggingFaceImageModel` enum directly is refused with a pointer to the correct pattern.

Async `generate()` routes the inference call through `asyncio.to_thread` so the event loop stays free.

In [ ]:
import asyncio
from aimu import aio, image_client, HuggingFaceImageModel

sync_client = image_client(HuggingFaceImageModel.SD_1_5)
async_client = aio.image_client(sync_client)

async def make_two():
    a, b = await asyncio.gather(
        async_client.generate("a watercolor of a fox", width=512, height=512, num_inference_steps=20),
        async_client.generate("a watercolor of a deer", width=512, height=512, num_inference_steps=20),
    )
    return a, b

# In a Jupyter kernel:
fox, deer = await make_two()
display(fox)
display(deer)

Note: on a single GPU, the two `gather`-ed calls don't truly overlap on the device (CUDA streams + GIL serialise inference), but the event loop stays free for other coroutines (request handlers, I/O, etc.). For true overlap across GPUs, build separate sync clients per device and wrap each.